In [1]:
import numpy as np
import plotly.express as px
import balancepy as bp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
from balancepy.model_sim.peterka18 import Peterka18 as P18
from balancepy.model_sim.asslaender23 import Asslaender23 as A23
from pprint import pprint


In [2]:
from concurrent.futures import ProcessPoolExecutor

# Move this function to the top level
def _parameter_recovery_simulation(args):
    """
    Run a single model simulation.

    Args:
        args (tuple): A tuple containing (model_class, frequencies, weight, height).

    Returns:
        tuple: set_values, fit_result
    """
    model_class, freq, weight, height, params_randomization = args

    # Initialize the model instance dynamically
    model_set = model_class(weight, height)

    # Select random parameter values within bounds
    n = 0
    for _, param in model_set.params.items():
        if not param.fixed:
            lower_bound, upper_bound = param.bounds
            param.value = lower_bound + (upper_bound-lower_bound) * params_randomization[n]
            n+=1

    # Store the randomly set parameter values
    params_input = model_set.params.values(only_free=False)

    # Simulate model
    data_sim = model_set.frf(freq)

    if np.all(data_sim.gain < 5):
        # Initialize new model - with data_sim as fit reference 
        model_fit = model_class(weight, height, data_exp = data_sim)
        model_fit.fit()

        params_recovered = model_fit.params.values(only_free=False)

    else:
        params_recovered = np.full_like(params_input, np.nan)

    return params_input, params_recovered


def parameter_recovery_noNoise(model_class, frequencies, num_simulations=100):
    """
    Run parallel simulations for a given model class.

    Args:
        model_class (type): The class of the model to simulate (e.g., P18.P18).
        frequencies (array-like): The frequency range for the model.
        num_simulations (int): Number of simulations to run.

    Returns:
        tuple: model_params, fit_results, par_names
    """

    # Create sample model
    sample_model = model_class(50, 1.5)
    par_names = sample_model.params.names()

    # Random selection of weight and height for each simulation
    weights = np.random.randint(50, 100, size=num_simulations)
    heights = np.random.uniform(1.5, 2.0, size=num_simulations)
    
    # Random parameter values; value is defined relative to bounds as parameters may depend on weight and height
    # Generate a list of random values between 0 and 1 for each free parameter, for each simulation
    free_params = [name for name, param in sample_model.params.items() if not param.fixed]
    params_randomization = np.random.uniform(0, 1, size=(num_simulations, len(free_params)))

    args = [
        (model_class, frequencies, weights[i], heights[i], params_randomization[i, :])
        for i in range(num_simulations)
    ]
    
    # Run simulations sequentially for debugging
    # results = [_parameter_recovery_simulation(arg) for arg in args]

    # Run simulations in parallel
    with ProcessPoolExecutor() as executor:
        results = list(executor.map(_parameter_recovery_simulation, args))


    # Extract the parameter values from the results
    params_input = np.array([result[0] for result in results])
    params_recovered = np.array([result[1] for result in results])


    return params_input, params_recovered, par_names


# Parameter recovery - Peterka 2018
The following section generates num_simulations artificial frequency response functions, and performs fits to test whether parameters are acurately recovered. The procedure is performed without adding noise to the data.

In [2]:
params_input, params_recovered, par_names = parameter_recovery_noNoise(
    model_class=P18,
    frequencies=np.arange(0.0165, 2.55, 0.033),
    num_simulations=500
)

In [3]:
# Assuming set_values is the array of parameter values used for plotting
num_params = len(par_names)

# Create subplots with two rows
fig = make_subplots(rows=3, cols=(num_params + 2) // 3, subplot_titles=par_names)

# Generate scatter plots for each parameter
for idx, param_name in enumerate(par_names):
    row = idx // ((num_params + 2) // 3) + 1
    col = idx % ((num_params + 2) // 3) + 1
    # Scatter plot of input vs recovered values
    fig.add_trace(
        go.Scatter(x=params_input[:, idx], y=params_recovered[:, idx], mode='markers', name=param_name),
        row=row, col=col
    )
    # Add identity line
    min_val = min(params_input[:, idx].min(), params_recovered[:, idx].min())
    max_val = max(params_input[:, idx].max(), params_recovered[:, idx].max())
    fig.add_trace(
        go.Scatter(
            x=[min_val, max_val],
            y=[min_val, max_val],
            mode='lines',
            line=dict(color='black', dash='dash'),
            showlegend=False
        ),
        row=row, col=col
    )

# Update layout
fig.update_layout(height=800, width=800, title_text="Parameter recovery", showlegend=False)

# Add a single shared x and y axis label as annotations
fig.add_annotation(
    text="input values",
    x=0.5, y=-0.03, xref="paper", yref="paper",
    showarrow=False, font=dict(size=18), xanchor="center", yanchor="top"
)
fig.add_annotation(
    text="recovered values",
    x=-0.02, y=0.5, xref="paper", yref="paper",
    showarrow=False, font=dict(size=18), textangle=-90, xanchor="right", yanchor="middle"
)

# Show the plot
fig.show()


# Parameter recovery - Asslaender 2023
The following section generates num_simulations artificial frequency response functions, and performs fits to test whether parameters are acurately recovered. The procedure is performed without adding noise to the data.

In [4]:
params_input, params_recovered, par_names = parameter_recovery_noNoise(
    model_class=A23,
    frequencies=np.arange(0.05, 2.55, 0.1),
    num_simulations=500
)

In [5]:
# Assuming set_values is the array of parameter values used for plotting
num_params = len(par_names)

# Create subplots with three rows
fig = make_subplots(rows=3, cols=(num_params + 2) // 3, subplot_titles=par_names)

# Generate scatter plots for each parameter
for idx, param_name in enumerate(par_names):
    row = idx // ((num_params + 2) // 3) + 1
    col = idx % ((num_params + 2) // 3) + 1
    # Scatter plot of input vs recovered values
    fig.add_trace(
        go.Scatter(x=params_input[:, idx], y=params_recovered[:, idx], mode='markers', name=param_name),
        row=row, col=col
    )
    # Add identity line
    min_val = min(params_input[:, idx].min(), params_recovered[:, idx].min())
    max_val = max(params_input[:, idx].max(), params_recovered[:, idx].max())
    fig.add_trace(
        go.Scatter(
            x=[min_val, max_val],
            y=[min_val, max_val],
            mode='lines',
            line=dict(color='black', dash='dash'),
            showlegend=False
        ),
        row=row, col=col
    )

# Update layout
fig.update_layout(height=800, width=800, title_text="Parameter recovery", showlegend=False)

# Add a single shared x and y axis label as annotations
fig.add_annotation(
    text="input values",
    x=0.5, y=-0.03, xref="paper", yref="paper",
    showarrow=False, font=dict(size=18), xanchor="center", yanchor="top"
)
fig.add_annotation(
    text="recovered values",
    x=-0.02, y=0.5, xref="paper", yref="paper",
    showarrow=False, font=dict(size=18), textangle=-90, xanchor="right", yanchor="middle"
)

# Show the plot
fig.show()